In [1]:
import example_loader as el
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml

In [28]:
# ways forward: maybe the +1 is wrong; I'm reducing vars (as I don't have the slacks)
# the negated rows may always be slacks on constraint >=
# is there one row or column that I must always negate in order for this to work?
# should I try the <= standardization once more?

import importlib
importlib.reload(gu)

def run_validation(instances):
    for instance in instances:
        model: gp.Model = instance.as_gurobi_model()
        print("Running model:", model.ModelName)
        model.params.LogToConsole = 0
        model.params.Method = 1
        gu.standardize_lt_to_gt(model)
        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(model)
        model.optimize()
        assert model.Status == gp.GRB.OPTIMAL
        basis = gu.read_basis(model)
        tableau, col_to_var, negated_rows = gu.read_tableau(model, basis)
        # tableau[negated_rows, :] *= -1.0
        gu.validate_corner(model, basis, tableau, col_to_var)

# test the cuts on simple examples:
run_validation(list(el.get_instances().values())[3:])

Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Running model: Book example 6.3
   Negated 3 constraints on Book example 6.3
   Relaxed 3 variables on Book example 6.3
   Failed validation! 0 <gurobi.Constr r3_rev> -1.0 x1 + -0.5 x3 [0.50801784 0.49732739 0.99465478] >= -1.0
   Failed validation! 1 <gurobi.Constr r4_rev> x1 + -1.0 x2 + -1.0 x3 [0.49732739 0.50801784 0.99465478] >= -1.0
